# Dimensión Novedad (dim_novedad) - Fast and Safe ETL

# Imports

In [26]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import String, create_engine, text

# Database connections and configuration

In [27]:
def load_config(path='../config.yml'):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def build_sqlalchemy_url(connection_config):
    return (
        f"{connection_config['drivername']}://{connection_config['user']}:{connection_config['password']}"
        f"@{connection_config['host']}:{connection_config['port']}/{connection_config['dbname']}"
    )


config = load_config('../config.yml')
config_co = config['CO_SA']
config_etl = config['ETL_PRO']

url_co = build_sqlalchemy_url(config_co)
url_etl = build_sqlalchemy_url(config_etl)

co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Database investigation and data loading

In [28]:
# Test de conexión y exploración de tablas de novedades
try:
    query_prueba = "SELECT * FROM public.mensajeria_novedadesservicio LIMIT 1;"
    df_test = pd.read_sql_query(query_prueba, co_sa)
    print("Conexión operacional verificada con éxito.")
    print("Columnas en mensajeria_novedadesservicio:", df_test.columns.tolist())
    display(df_test)
except Exception as e:
    print("Error al intentar leer de la base de datos:")
    print(e)
    print("\nNota: Si sale error de autenticación, copia 'config_fill.yml' a 'config.yml' con tus datos reales.")


Conexión operacional verificada con éxito.
Columnas en mensajeria_novedadesservicio: ['id', 'fecha_novedad', 'tipo_novedad_id', 'descripcion', 'servicio_id', 'es_prueba', 'mensajero_id']


,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7


In [29]:
# Inspeccionar las tablas de novedades disponibles
try:
    query_novedad = "SELECT * FROM public.mensajeria_novedadesservicio LIMIT 5;"
    df_novedad = pd.read_sql_query(query_novedad, co_sa)
    print("Tabla de novedades encontrada:")
    print("Columnas:", df_novedad.columns.tolist())
    print("Total de registros:", len(df_novedad))
    display(df_novedad)
except Exception as e:
    print("Error al leer mensajeria_novedadesservicio:")
    print(e)
    print("Intentando otras posibles tablas de novedades...")


Tabla de novedades encontrada:
Columnas: ['id', 'fecha_novedad', 'tipo_novedad_id', 'descripcion', 'servicio_id', 'es_prueba', 'mensajero_id']
Total de registros: 5


,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


# Extract

In [30]:
query_novedades = """
SELECT DISTINCT
    id AS id_novedad_origen,
    tipo_novedad_id AS categoria_novedad,
    descripcion AS descripcion_novedad
FROM public.mensajeria_novedadesservicio
ORDER BY id;
"""

df_novedades = pd.read_sql_query(query_novedades, co_sa)
print(f"Tipos de novedades extraídos: {len(df_novedades)}")
display(df_novedades.head())

Tipos de novedades extraídos: 5208


,id_novedad_origen,categoria_novedad,descripcion_novedad
0,4,1,A
1,5,1,Halo
2,6,1,A
3,7,1,B
4,8,1,A


# Transform

In [31]:
dim_novedad = df_novedades.copy()

dim_novedad['categoria_novedad'] = (
    dim_novedad['categoria_novedad']
    .fillna('SIN CATEGORÍA')
    .astype(str)
    .str.strip()
    .str.upper()
)

dim_novedad['descripcion_novedad'] = (
    dim_novedad['descripcion_novedad']
    .fillna('SIN DESCRIPCIÓN')
    .astype(str)
    .str.strip()
    .str.upper()
)

dim_novedad['descripcion'] = dim_novedad['descripcion_novedad'].str.slice(0, 500)

dim_novedad = dim_novedad.drop(columns=['descripcion_novedad'])

dim_novedad = dim_novedad.drop_duplicates(
    subset=['categoria_novedad', 'descripcion']
).reset_index(drop=True)

dim_novedad['saved_novedad'] = date.today()

dim_novedad = dim_novedad.sort_values(
    by=['categoria_novedad', 'descripcion']
).reset_index(drop=True)

print(f"Tipos únicos de novedades: {len(dim_novedad)}")
display(dim_novedad.head())


Tipos únicos de novedades: 3918


,id_novedad_origen,categoria_novedad,descripcion,saved_novedad
0,1144,1,# DE TRASLADO TB3 75942,2026-06-24
1,719,1,(EL SEÑOR LIBARDO APENAS ENVÍO LA FACTURA),2026-06-24
2,2301,1,"1, COMPRA EXITOSA PROVEEDOR EL ARRANQUE, BOMBI...",2026-06-24
3,1459,1,"1, COMPRA EXITOSA, PROVEEDOR UNI TORNILLOS SAS...",2026-06-24
4,3118,1,"1, COMPRA EXITOSA, PROVEEDOR UNITORNILLOS SAS,...",2026-06-24


# Validation

In [32]:
print('--- VALIDACIÓN dim_novedad ---')
print('\nValores nulos:')
print(dim_novedad.isnull().sum())

print('\nDuplicados por categoría y descripción:')
print(
    dim_novedad.duplicated(
        subset=['categoria_novedad', 'descripcion']
    ).sum()
)

print('\nCantidad de novedades por categoría:')
print(dim_novedad['categoria_novedad'].value_counts())

print('\nTipos de datos:')
print(dim_novedad.dtypes)

--- VALIDACIÓN dim_novedad ---

Valores nulos:
id_novedad_origen    0
categoria_novedad    0
descripcion          0
saved_novedad        0
dtype: int64

Duplicados por categoría y descripción:
0

Cantidad de novedades por categoría:
categoria_novedad
1    2894
2    1024
Name: count, dtype: int64

Tipos de datos:
id_novedad_origen     int64
categoria_novedad       str
descripcion             str
saved_novedad        object
dtype: object


# Load

In [33]:
crear_dim_novedad = """
CREATE TABLE public.dim_novedad (
    id_novedad SERIAL PRIMARY KEY,
    id_novedad_origen INTEGER NOT NULL UNIQUE,
    categoria_novedad VARCHAR(100) NOT NULL,
    descripcion VARCHAR(500) NOT NULL,
    saved_novedad DATE NOT NULL
);
"""

with etl_conn.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS public.dim_novedad;"))
    ##conn.execute(text(crear_dim_novedad))

dim_novedad.to_sql(
    name='dim_novedad',
    con=etl_conn,
    schema='public',
    if_exists='append',
    index_label='key_dim_novedad',
    dtype={
        'descripcion': String(500)
    },
    method='multi'
)

print(f"Dimensión cargada correctamente: {len(dim_novedad)} registros.")


Dimensión cargada correctamente: 3918 registros.


# Visualization

In [34]:
query_verificacion = """
SELECT
    id_novedad,
    id_novedad_origen,
    categoria_novedad,
    descripcion,
    saved_novedad
FROM public.dim_novedad
ORDER BY id_novedad;
"""

# df_dim_novedad = pd.read_sql_query(query_verificacion, etl_conn)
# print(f"Registros en dim_novedad: {len(df_dim_novedad)}")
# display(df_dim_novedad)
